# Reto 1 — Zonas de falla en la red eléctrica de Costa Rica como Max-Cut

**Quantathon CR 2026 · Challenge 1** · Dojo Coding · UCR · OQI · Quantinuum

Este cuaderno cuenta la solución completa de punta a punta: partimos de los
**datos abiertos reales de la red de transmisión del ICE**, modelamos el
particionamiento en zonas de falla como **Max-Cut**, lo formulamos como
**QUBO**, lo resolvemos con **QAOA** (híbrido cuántico-clásico, Qiskit + emuladores
H-series de Quantinuum vía Nexus) y lo comparamos honestamente contra las
líneas base clásicas más fuertes (exacto, **Goemans-Williamson**, greedy,
recocido simulado) — escalando después la comparación **a todo el país**
(68 subestaciones).

> Ejecutar antes `python reproduce.py` (una vez) para generar los registros y
> figuras que este cuaderno consume; todas las celdas vivas usan instancias
> pequeñas y corren en segundos.

## 1 · El problema: aislar fallas sin apagar el país

Cuando una falla ocurre en una red de transmisión, los operadores necesitan
**particionar la red en islas** que contengan el problema minimizando los
circuitos que hay que abrir entre zonas — cada corredor abierto es capacidad
de transporte perdida. Modelamos la red como un grafo ponderado
$G = (V, E, w)$: los nodos son subestaciones, las aristas corredores de
transmisión, y el peso $w_{ij}$ el número de circuitos paralelos del corredor.
Partir la red en dos zonas maximizando el peso **cortado**… es exactamente
**Max-Cut**, un problema NP-hard.

La instancia estrella (**cr8**) se deriva de forma determinista y auditable de
los datos abiertos del Grupo ICE (70 subestaciones, 102 circuitos;
`data/README.md` documenta cada paso). ODS relevantes: **7** (energía asequible
y no contaminante), **9** (industria, innovación e infraestructura) y
**13** (acción por el clima).

In [ ]:
import json
from pathlib import Path

from reto1.instances import load_instance

ROOT = Path("..").resolve()
inst = load_instance(ROOT / "data" / "cr8-uniforme.json")
record = json.loads((ROOT / "data" / "cr8-uniforme.json").read_text())

print(f"instancia : {inst.name}")
print(f"nodos     : {inst.n_nodes}")
for k in sorted(record["nodos"], key=int):
    print(f"   {k}: {record['nodos'][k]}")
print(f"aristas   : {len(inst.edges)}  (peso = circuitos paralelos)")
print(f"óptimo    : {inst.optimum}  (probado por {' + '.join(inst.methods)})")
print(f"digest    : {inst.digest[:16]}…  (la identidad congelada de la instancia)")

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

g = nx.Graph()
for i, j, w in inst.edges:
    g.add_edge(i, j, weight=w)
labels = {int(k): v for k, v in record["nodos"].items()}
pos = nx.kamada_kawai_layout(g)
fig, ax = plt.subplots(figsize=(7, 5))
nx.draw_networkx(g, pos, labels=labels, node_color="#a6cbe3", node_size=1600,
                 font_size=7, edgecolors="#1a1a1a", ax=ax)
nx.draw_networkx_edge_labels(g, pos, {(u, v): d["weight"]
                                      for u, v, d in g.edges(data=True)},
                             font_size=8, ax=ax)
ax.set_title("cr8: corredor más denso de la red ICE (GAM)")
ax.axis("off")
plt.show()

## 2 · De Max-Cut a QUBO

Con variables binarias $x_i \in \{0, 1\}$ (la zona de cada subestación), el
corte es

$$\mathrm{cut}(x) = \sum_{(i,j) \in E} w_{ij}\,(x_i + x_j - 2 x_i x_j),$$

y **maximizarlo** equivale a **minimizar** la energía QUBO
$E(x) = x^\top Q x$ con

$$Q_{ii} = -\sum_{j} w_{ij}, \qquad Q_{ij} = Q_{ji} = w_{ij}.$$

Max-Cut no tiene restricciones, así que la QUBO no necesita términos de
penalización: $E(x) = -\mathrm{cut}(x)$ **exactamente**. Lo verificamos de
forma exhaustiva (los $2^n$ estados) sobre la instancia real:

In [ ]:
import itertools

import numpy as np

from reto1.maxcut import cut_value
from reto1.qubo import qubo_matrix, qubo_energy

q = qubo_matrix(inst.n_nodes, inst.edges)
for x_bits in itertools.product((0, 1), repeat=inst.n_nodes):
    x = np.array(x_bits)
    assert qubo_energy(q, x) == -cut_value(inst.edges, x_bits)
print(f"E(x) = -cut(x) verificado en los {2**inst.n_nodes} estados de cr8 ✓")
print(f"mínimo de E = {min(qubo_energy(q, np.array(x)) for x in itertools.product((0,1), repeat=inst.n_nodes))} = -óptimo ({inst.optimum}) ✓")

## 3 · Líneas base clásicas (la vara honesta)

La rúbrica exige comparar contra el clásico **más fuerte** disponible:

- **Fuerza bruta / doble ancla** — el óptimo exacto está congelado en cada
  instancia, probado por dos códigos independientes.
- **Goemans-Williamson (1995)** — relajación SDP + redondeo por hiperplanos
  aleatorios, razón garantizada ≥ 0.878. El óptimo del SDP es además una
  **cota superior rigurosa** del corte máximo (ancla de cordura).
- **Greedy** (~0.5) y **recocido simulado** (multi-semilla).

In [ ]:
from reto1.gw import goemans_williamson
from reto1.maxcut import greedy_cut, simulated_annealing

gw = goemans_williamson(inst.n_nodes, inst.edges, seed=0)
greedy_v, _ = greedy_cut(inst.n_nodes, inst.edges)
sa_v, _ = simulated_annealing(inst.n_nodes, inst.edges, seed=0)
print(f"óptimo exacto : {inst.optimum}")
print(f"GW            : {gw.value}  (cota SDP = {gw.sdp_bound:.2f} ≥ óptimo)")
print(f"greedy        : {greedy_v}")
print(f"recocido sim. : {sa_v}")
print(f"razón GW      : {gw.value / inst.optimum:.4f}  (garantía teórica ≥ 0.878)")

## 4 · QAOA: el algoritmo cuántico híbrido

QAOA (Farhi et al. 2014) prepara el estado variacional

$$|\gamma, \beta\rangle = \prod_{l=1}^{p} e^{-i \beta_l B}\, e^{-i \gamma_l C}\, |+\rangle^{\otimes n},$$

alternando $p$ capas de la **fase de costo** ($C$, diagonal: RZZ por arista) y
del **mixer** ($B = \sum_i X_i$: RX por qubit). Los ángulos $(\gamma, \beta)$
se optimizan **clásicamente** sobre la expectativa exacta del statevector
(COBYLA multi-arranque + warm start $p{-}1 \to p$); el circuito optimizado se
**muestrea** después (Aer local con semilla, o el emulador H2 real vía Nexus).
Cada corte reportado se **recomputa clásicamente** desde el bitstring — nunca
confiamos en la contabilidad del backend.

La métrica oficial es la razón de aproximación
$r = \langle \mathrm{cut} \rangle / \mathrm{óptimo}$.

In [ ]:
from reto1.qaoa import build_qaoa_circuit

demo = load_instance(ROOT / "data" / "cr6-uniforme.json")
qc = build_qaoa_circuit(demo.n_nodes, demo.edges, gammas=[0.55], betas=[0.31])
print(qc.draw(output="text", fold=100))

In [ ]:
from reto1.qaoa import run_qaoa

res = run_qaoa(demo.n_nodes, demo.edges, optimum=demo.optimum, p=1, seed=0,
               shots=2048)
print(f"cr6-uniforme p=1: r_esperado = {res.ratio_expected:.4f}")
print(f"criterio oficial del reto (r ≥ 0.6 en 6 nodos, p=1): "
      f"{'CUMPLIDO ✓' if res.ratio_expected >= 0.6 else 'NO'}")
print(f"mejor muestra alcanza el óptimo: "
      f"{'sí' if res.best_sampled_cut == demo.optimum else 'no'}")

## 5 · Resultados completos (estadística multi-semilla + emuladores reales)

`reproduce.py` corre cada configuración con **5 semillas** y reporta media ± σ
— nunca una corrida suelta. Las mismas configuraciones se ejecutaron en los
**emuladores H-series de Quantinuum** vía Nexus (H2-1LE sin ruido y
H2-Emulator con el modelo de ruido realista), reutilizando los ángulos
optimizados localmente (split híbrido documentado).

In [ ]:
local = {}
for path in sorted((ROOT / "runs" / "local").glob("*.json")):
    r = json.loads(path.read_text())
    if r.get("qaoa_local"):
        local[r["instance"]] = r["qaoa_local"]

print(f"{'instancia':16} {'p':>2} {'r (media ± σ)':>20} {'r mejor muestra':>16}")
for name, sweep in sorted(local.items()):
    for p in sorted(sweep, key=int):
        re_ = sweep[p]["ratio_expected"]
        rb = sweep[p]["ratio_best"]
        print(f"{name:16} {p:>2} {re_['mean']:>12.4f} ± {re_['std']:.4f} "
              f"{rb['mean']:>16.4f}")

In [ ]:
from IPython.display import Image, display

display(Image(str(ROOT / "figures" / "r-vs-p-cr8.png"), width=640))
display(Image(str(ROOT / "figures" / "ruido-h2.png"), width=640))

La brecha ideal-vs-ruidoso del emulador H2 es pequeña a estas profundidades
(circuitos cortos, 6-14 qubits) — y **el análisis de ruido es obligatorio**
para reportar resultados de hardware sin caer en una red flag de la rúbrica.

## 6 · Escalado a todo el país (68 subestaciones)

El mismo criterio determinista que construyó cr8 crece una **familia anidada
de corredores** cr8 ⊂ cr12 ⊂ cr16 ⊂ cr20 ⊂ cr26 ⊂ … ⊂ **cr68 = la red
nacional completa**.

- Hasta **20 nodos**: óptimo probado (doble ancla) → QAOA con estadística
  completa.
- **26 nodos** es el techo del emulador H2; **20** el de nuestra optimización
  local de ángulos (statevector denso).
- Más allá: **solo clásico**, reportando el intervalo honesto
  **[mejor corte hallado, cota superior SDP]** — nunca un óptimo fingido.

In [ ]:
open_reports = {}
for path in sorted((ROOT / "runs" / "local").glob("*.json")):
    r = json.loads(path.read_text())
    if r.get("classical_open"):
        open_reports[r["instance"]] = r["classical_open"]

print(f"{'instancia':16} {'n':>3} {'mejor hallado':>14} {'cota SDP':>9} {'r ≥':>7}")
for name, rep in sorted(open_reports.items(), key=lambda kv: kv[1]["n_nodes"]):
    print(f"{name:16} {rep['n_nodes']:>3} "
          f"{rep['best_found']:>7} ({rep['best_method']}) "
          f"{rep['gw']['sdp_bound']:>9.1f} {rep['ratio_lower_bound']:>7.4f}")

In [ ]:
display(Image(str(ROOT / "figures" / "mapa-nacional.png"), width=760))
display(Image(str(ROOT / "figures" / "escalado-nacional.png"), width=680))

## 7 · Port de paridad a Guppy (SDK recomendado)

El circuito estrella se portó a **Guppy** y se ejecutó en el emulador Selene
(Quest): mismo ansatz, RZZ descompuesto exactamente como CX·RZ·CX, ángulos en
semigiros (convención tket, fijada empíricamente). La media muestreada debe
caer dentro de 4σ de la expectativa exacta — y cae dentro de 1σ.

In [ ]:
guppy_record = json.loads(
    (ROOT / "runs" / "guppy" / "cr8-uniforme-p1-s0.json").read_text())
par = guppy_record["parity"]
print(f"SDK        : {guppy_record['sdk']}")
print(f"⟨cut⟩      : {guppy_record['stats']['mean_cut']:.4f}")
print(f"exacto     : {par['expected_cut_exact']:.4f} ± {par['sigma_of_mean']:.4f}")
print(f"desviación : {par['deviation']:.4f}  ({par['criterion']})")
print(f"paridad    : {'OK ✓' if par['ok'] else 'FALLIDA'}")
print(f"r_mean     : {guppy_record['stats']['ratio_mean']:.4f}")

## 8 · Limitaciones honestas (obligatorio — y con gusto)

1. **QAOA no supera a Goemans-Williamson en Max-Cut.** La garantía de QAOA a
   p=1 (0.6924) es estrictamente menor que la de GW (0.878); en nuestras
   instancias GW además **encuentra el óptimo exacto**. A estas escalas
   (6-30 nodos) un solver exacto responde en milisegundos: el valor del
   ejercicio es el flujo híbrido verificado, no una ventaja cuántica.
2. **Split híbrido:** los ángulos se optimizan sobre el statevector local
   exacto; el emulador muestrea (no optimiza en el loop). Estándar a escala
   NISQ, y documentado.
3. **El emulador remoto no expone semilla por shot** → sus corridas son
   estadística de muestreo, no réplicas exactas.
4. **Los pesos `voltaje` son un proxy documentado** (suma de kV) — los datos
   abiertos no traen capacidad MVA ni caso de flujo.
5. **Más allá de 20/26 qubits no hay pata cuántica** (límite de optimización
   local / techo del emulador H2): la escalera nacional se reporta con el
   intervalo [mejor clásico, cota SDP], sin extrapolar afirmaciones cuánticas.
6. **La factibilidad física del islanding** (balance generación/carga por
   isla, criterio N-1) no está codificada en Max-Cut plano; corresponde a la
   extensión oficial de *constraint mixers*.

## 9 · Reproducibilidad

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
python reproduce.py            # todas las cifras y figuras (escalera incluida)
python reproduce.py --quick    # smoke rápido

# emuladores Quantinuum (credenciales del evento) — cacheado en runs/nexus/
qnx login && uv run python scripts/run_h2_emulator.py --device H2-1LE

# port Guppy/Selene (grupo de dependencias `entregables`)
uv run python scripts/run_guppy_qaoa.py

# tests
pytest
```

Cada instancia y cada registro llevan **digest SHA-256 canónico**; los datos
crudos del ICE están congelados en `data/raw/`; el cuaderno completo se
re-ejecuta con `jupyter nbconvert --execute`.